# Sandbox

In [88]:
import warnings

import gspread
import pandas as pd
from category_encoders import JamesSteinEncoder, MEstimateEncoder
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer

warnings.filterwarnings('ignore')

## Data Extraction

### Google Sheets Backlog

In [89]:
# 1. Enable Sheets API in GCP concole
# 2. Create OAuth JSON in GCP console
# 3. Add personal email as test user
# 4. Authorize application in the browser

gc = gspread.oauth()

In [90]:
sh = gc.open('Backlog')
worksheet = sh.get_worksheet(0)

data = worksheet.get_all_records()

### Cleaning

In [91]:
df = pd.DataFrame(data)

In [92]:
df['Nota'] = pd.to_numeric(df['Nota'], errors='coerce')

In [93]:
df['Franquia'] = df['Franquia'].fillna('Independente').replace('—', 'Independente')
df['Franquia'].value_counts()

Franquia
Independente          33
Resident Evil          9
Souls                  6
Final Fantasy          4
Onimusha               4
Pokémon                3
Zelda                  3
Metro                  3
Monster Hunter         3
LEGO                   2
Spider Man             2
God of War             2
Shin Megami            2
Kingdom Come           2
Tales of               1
Hades                  1
Like a Dragon          1
Star Wars Jedi         1
TLOU                   1
Kingdom Hearts         1
Dead Space             1
Ghost of Tsushima      1
BioShock               1
Granblue Fantasy       1
Ghost Recon            1
Control                1
Darksiders             1
Dragon Quest           1
Ordem Paranormal       1
Fallout                1
Hollow Knight          1
Hotline Miami          1
Titan Quest            1
Warhammer 40k          1
Yakuza                 1
Octopath Traveler      1
Life is Strange        1
Citizen Sleeper        1
Ace Combat             1
The Witcher     

In [94]:
df['Desenvolvedora'].fillna('Desconhecida').replace('—', 'Desconhecida')
df['Desenvolvedora'].value_counts()

Desenvolvedora
Capcom           17
Square Enix       6
FromSoftware      6
Nintendo          3
Atlus             3
                 ..
Subset Games      1
Team Meat         1
Cold Symmetry     1
PlatinumGames     1
Ska Studios       1
Name: count, Length: 69, dtype: int64

In [95]:
df_finished = df[df['Status'] == '3. Finalizados'].copy()
df_finished = df_finished.dropna(subset=['Nota'])
df_finished = df_finished.reset_index(drop=True)

df_finished.head()

,Jogo,Plataforma,Status,Gênero,Multiplayer,Nota,Review,Data,Franquia,Desenvolvedora,Duração
0,Resident Evil 6,PC,3. Finalizados,"Survival Horror, Coop",TRUE,7.0,Dá pra entender o porquê de dizerem que é o pi...,31/05/2026,Resident Evil,Capcom,Médio (15–40h)
1,Pokémon Brilliant Diamond,Switch,3. Finalizados,JRPG,FALSE,9.0,"Não é o platinum, mas ainda é o diamond",22/05/2026,Pokémon,Game Freak,Médio (15–40h)
2,Final Fantasy 16,PC,3. Finalizados,JRPG,FALSE,9.0,"Combate delicioso, com uma história excelente",17/05/2026,Final Fantasy,Square Enix,Longo (40h+)
3,Resident Evil 5,PC,3. Finalizados,"Coop, Survival Horror",TRUE,7.0,Divertido em coop,03/04/2026,Resident Evil,Capcom,Curto (até 15h)
4,Marvel’s Spider-Man: Miles Morales,PC,3. Finalizados,Ação/Aventura,FALSE,8.0,"Jogo divertido, melhor que o primeiro",27/03/2026,Spider Man,Insomniac,Curto (até 15h)


In [96]:
df_backlog = df[
    df['Status'].isin([
        '2. Próximos',
        '4. Backlog',
        '5. Rejogar',
        '6. Dar Outra Chance',
    ])
].copy()
df_backlog = df_backlog.reset_index(drop=True)

df_backlog.head()

,Jogo,Plataforma,Status,Gênero,Multiplayer,Nota,Review,Data,Franquia,Desenvolvedora,Duração
0,Dark Souls 3 (replay),PC,2. Próximos,"Coop, Soulslike",TRUE,11.0,,,Souls,FromSoftware,Médio (15–40h)
1,Resident Evil,PC,2. Próximos,Survival Horror,FALSE,NaN,,,Resident Evil,Capcom,Curto (até 15h)
2,Onimusha: Warlords,PC,2. Próximos,"Ação/Aventura, Hack and Slash",FALSE,NaN,,,Onimusha,Capcom,Curto (até 15h)
3,Onimusha 2: Samurai's Destiny,PC,2. Próximos,"Ação/Aventura, Hack and Slash",FALSE,NaN,,,Onimusha,Capcom,Curto (até 15h)
4,Kingdom Come: Deliverance II,PC,4. Backlog,RPG,FALSE,NaN,,,Kingdom Come,Warhorse Studios,Longo (40h+)


## Feature Engineering

### Genre

In [97]:
df_finished['genres_list'] = df_finished['Gênero'].str.split(', ')
mlb = MultiLabelBinarizer()
genre_features = pd.DataFrame(
    mlb.fit_transform(df_finished['genres_list']), columns=mlb.classes_
)

df_train = pd.concat([df_finished, genre_features], axis=1)

df_train.head()

,Jogo,Plataforma,Status,Gênero,Multiplayer,Nota,Review,Data,Franquia,Desenvolvedora,...,Hack and Slash,JRPG,Plataforma / Ação 2D,Puzzle,RPG,Roguelike,Shooter / Tiro,Simulação / Arcade,Soulslike,Survival Horror
0,Resident Evil 6,PC,3. Finalizados,"Survival Horror, Coop",TRUE,7.0,Dá pra entender o porquê de dizerem que é o pi...,31/05/2026,Resident Evil,Capcom,...,0,0,0,0,0,0,0,0,0,1
1,Pokémon Brilliant Diamond,Switch,3. Finalizados,JRPG,FALSE,9.0,"Não é o platinum, mas ainda é o diamond",22/05/2026,Pokémon,Game Freak,...,0,1,0,0,0,0,0,0,0,0
2,Final Fantasy 16,PC,3. Finalizados,JRPG,FALSE,9.0,"Combate delicioso, com uma história excelente",17/05/2026,Final Fantasy,Square Enix,...,0,1,0,0,0,0,0,0,0,0
3,Resident Evil 5,PC,3. Finalizados,"Coop, Survival Horror",TRUE,7.0,Divertido em coop,03/04/2026,Resident Evil,Capcom,...,0,0,0,0,0,0,0,0,0,1
4,Marvel’s Spider-Man: Miles Morales,PC,3. Finalizados,Ação/Aventura,FALSE,8.0,"Jogo divertido, melhor que o primeiro",27/03/2026,Spider Man,Insomniac,...,0,0,0,0,0,0,0,0,0,0


### Franchise

In [98]:
# On the final project, solve the leakage problem

jse = JamesSteinEncoder()
franchise_features = jse.fit_transform(
    df_finished['Franquia'], df_finished['Nota'].astype(int)
)

df_train['franchise_score'] = franchise_features

df_train['franchise_score'].head()

0    8.247036
1    9.000000
2    8.580440
3    8.247036
4    8.000000
Name: franchise_score, dtype: float64

### Developer

In [99]:
# On the final project, solve the leakage problem

mee = MEstimateEncoder(m=10)
developer_features = mee.fit_transform(
    df_finished['Desenvolvedora'], df_finished['Nota'].astype(int)
)

df_train['developer_score'] = developer_features

df_train['developer_score'].head()

0    8.233918
1    8.313131
2    8.246032
3    8.233918
4    8.203704
Name: developer_score, dtype: float64

## Model Training and Testing

In [100]:
genre_finished = pd.DataFrame(
    mlb.transform(df_finished['genres_list']),
    columns=mlb.classes_,
    index=df_finished.index,
)
franchise_finished = jse.fit_transform(
    df_finished[['Franquia']], df_finished['Nota'].astype(int)
)
developer_finished = mee.fit_transform(
    df_finished[['Desenvolvedora']], df_finished['Nota'].astype(int)
)

In [101]:
X_train = pd.concat([genre_finished, franchise_finished, developer_finished], axis=1)
y_train = df_finished['Nota']

In [102]:
loo = LeaveOneOut()
model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=loo)
model.fit(X_train, y_train)

loo_scores = cross_val_score(
    RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
    X_train,
    y_train,
    cv=loo,
    scoring='neg_mean_absolute_error',
)
print(
    f'LOO MAE:  {-loo_scores.mean():.3f} ± {loo_scores.std():.3f}  |  Best alpha: {model.alpha_}'
)

LOO MAE:  0.641 ± 0.600  |  Best alpha: 0.01


In [103]:
df_backlog['genres_list'] = df_backlog['Gênero'].str.split(', ')

genre_backlog = pd.DataFrame(
    mlb.transform(df_backlog['genres_list']),
    columns=mlb.classes_,
    index=df_backlog.index,
)
franchise_backlog = jse.transform(df_backlog[['Franquia']])
developer_backlog = mee.transform(df_backlog[['Desenvolvedora']])

X_backlog = pd.concat([genre_backlog, franchise_backlog, developer_backlog], axis=1)

df_backlog['Nota Prevista'] = model.predict(X_backlog).clip(1, 10).round(2)

df_backlog[['Jogo', 'Franquia', 'Gênero', 'Status', 'Nota Prevista']].sort_values(
    'Nota Prevista', ascending=False
).reset_index(drop=True)

,Jogo,Franquia,Gênero,Status,Nota Prevista
0,Dungeon Rats,Independente,CRPG,4. Backlog,9.71
1,Outer Wilds,Independente,Exploração / Mistério,4. Backlog,9.71
2,Dark Souls II (replay),Souls,Soulslike,5. Rejogar,9.68
3,Dark Souls (replay),Souls,Soulslike,5. Rejogar,9.68
4,Divinity: Original Sin 2,Divinity,CRPG,6. Dar Outra Chance,9.53
...,...,...,...,...,...
56,Horizon Forbidden West,Horizon,"Ação/Aventura, RPG",6. Dar Outra Chance,7.82
57,Alan Wake,Alan Wake,"Survival Horror, Aventura Narrativa",6. Dar Outra Chance,7.71
58,Citizen Sleeper 2: Starward Vector,Citizen Sleeper,"RPG, Aventura Narrativa",4. Backlog,7.65
59,A Way Out,Independente,"Aventura Narrativa, Coop",6. Dar Outra Chance,7.40
